#Loading and Normailzing CIFAR10

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.v2 as v2

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=4)

100%|██████████| 170M/170M [24:56<00:00, 114kB/s]


#Constructing CNN Classifer and Training it

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

class Net(nn.Module):
  def __init__(self):
    super().__init__()

    self.network= nn.Sequential(
        #First layer takes the 3 channel RGB and returns 6 filter/feature.
        nn.Conv2d(3,6,5),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        #Second layer takes the 6 channel output by the first conv layer
        nn.Conv2d(6,16,5),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        #Flatten to 1d vector
        nn.Flatten(1),

        #In first layer of we have 120 neuron; input matrix containing 400 elemnets
        nn.Linear(16*5*5,120),
        nn.ReLU(),

        #Now second layer neuron is 84
        nn.Linear(120,84),
        nn.ReLU(),

        #Because we have 10 classes we need 10 neuron in output layer
        nn.Linear(84,10)

    )

  def forward(self,x):
    return self.network(x)

net=Net()

# Check for GPU and move model to device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
net.to(device)

#using stochastic gradient descent optimization and CE loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

# loop runs twice, we have batch size of 4 defined in 1st cell
for epoch in range (10):

  running_loss = 0.0
  for i, (inputs, labels) in enumerate(trainloader, 0):
    # get the inputs; data is a list of [inputs, labels]
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()
    output= net(inputs)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    if i % 2000 == 1999:    # print every 2000 mini-batches
        print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
        running_loss = 0.0

print('Finished Training')


[1,  2000] loss: 2.228
[1,  4000] loss: 1.932
[1,  6000] loss: 1.710
[1,  8000] loss: 1.594
[1, 10000] loss: 1.572
[1, 12000] loss: 1.497
[2,  2000] loss: 1.446
[2,  4000] loss: 1.434
[2,  6000] loss: 1.373
[2,  8000] loss: 1.338
[2, 10000] loss: 1.356
[2, 12000] loss: 1.308
[3,  2000] loss: 1.271
[3,  4000] loss: 1.270
[3,  6000] loss: 1.233
[3,  8000] loss: 1.206
[3, 10000] loss: 1.230
[3, 12000] loss: 1.193
[4,  2000] loss: 1.156
[4,  4000] loss: 1.166
[4,  6000] loss: 1.132
[4,  8000] loss: 1.119
[4, 10000] loss: 1.145
[4, 12000] loss: 1.112
[5,  2000] loss: 1.074
[5,  4000] loss: 1.090
[5,  6000] loss: 1.054
[5,  8000] loss: 1.048
[5, 10000] loss: 1.088
[5, 12000] loss: 1.046
[6,  2000] loss: 1.007
[6,  4000] loss: 1.033
[6,  6000] loss: 1.000
[6,  8000] loss: 0.994
[6, 10000] loss: 1.028
[6, 12000] loss: 0.998
[7,  2000] loss: 0.958
[7,  4000] loss: 0.984
[7,  6000] loss: 0.946
[7,  8000] loss: 0.946
[7, 10000] loss: 0.987
[7, 12000] loss: 0.944
[8,  2000] loss: 0.934
[8,  4000] 

#Testing our Model

In [7]:
with torch.no_grad():
    #iterate through a DataLoader for testing
    dataiter = iter(testloader)
    images, labels = next(dataiter)
    outputs = net(images)
    #give predicted class as the one having max value in dim=1
    _, predicted = torch.max(outputs, 1)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print('Ground Truth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))
print('Predicted:    ', ' '.join(f'{classes[predicted[j]]:5s}' for j in range(4)))

Ground Truth:  cat   ship  ship  plane
Predicted:     cat   ship  ship  plane
